# 04 — Create a Bedrock Managed Knowledge Base with OneDrive Connector

This notebook walks through the end-to-end flow of creating a **Bedrock Managed Knowledge Base (BMKB)** with **Microsoft OneDrive** as the data source. It ingests files from the personal drives of users in your Microsoft 365 tenant.

### What this notebook does

1. Stores your OneDrive credentials in AWS Secrets Manager
2. Creates the BMKB with all required IAM roles and policies
3. Ingests documents from OneDrive
4. Queries the KB using Retrieve and AgenticRetrieveStream APIs
5. Cleans up all resources

### Prerequisites

- A Microsoft 365 tenant and its **tenant ID** (Microsoft Entra / Azure AD directory ID)
- A registered Entra application with Microsoft Graph **application** permissions: `Files.Read.All`, `Sites.Read.All`, `User.Read.All` (add `GroupMember.Read.All` and SharePoint `Sites.FullControl.All` if you enable document-level access control)
  - Setup: [Set up Microsoft Entra App ID authentication for OneDrive](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-onedrive-entra-setup.html)
  - Alternative refresh-token flow: [Set up OAuth 2.0 authentication for OneDrive](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-onedrive-oauth2-setup.html)
- AWS credentials with Bedrock, IAM, and Secrets Manager permissions
- **Kernel:** Select `Python 3`

### Architecture

```
OneDrive (Microsoft 365) ──► BMKB (Bedrock-managed vector store) ──► Retrieve / AgenticRetrieveStream
        │                              │
        ├── Personal drives             ├── IAM Role (auto-created)
        ├── User email filters          ├── Secrets Manager (Entra/OAuth creds)
        └── Date / MIME / path filters  └── Smart Parsing
```

### Supported authentication

| Auth Type | When to use | Document-level ACLs |
|-----------|-------------|---------------------|
| `ENTRA_APP_ID` (recommended) | Crawl every user's OneDrive uniformly | Supported |
| `OAUTH2` | A single signed-in user can reach all the content you want to crawl | Not supported |

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

## Step 1 — Configuration

Update the values below to match your OneDrive environment.

In [ ]:
import boto3
import json
import sys
import time
import logging
import pprint

sys.path.insert(0, "../..")

# Clients
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration (update these) ─────────────────────────────────────────
knowledge_base_name = f'bmkb-onedrive-{suffix}'
knowledge_base_description = 'BMKB - OneDrive data source'

# OneDrive / Microsoft 365 settings
tenant_id = '<your-microsoft-365-tenant-id>'   # v4 UUID e.g. 12345678-1234-4abc-8def-1234567890ab
auth_type = 'ENTRA_APP_ID'                     # ENTRA_APP_ID (recommended) | OAUTH2

# Document-level access control (Entra App ID only)
acl_enabled = False
certificate_s3_bucket = '<your-cert-bucket>'   # bucket holding the .p12 bundle (only when acl_enabled=True)
certificate_s3_key    = 'certs/certificate.p12'

# Secret — set an existing ARN to skip creation, or leave None to create one below
existing_secret_arn = None
secret_name = f'bmkb-onedrive-creds-{suffix}'

# Entra App ID credentials (only used if creating a new secret with auth_type=ENTRA_APP_ID)
od_client_id            = '<your-client-id>'
od_client_secret        = '<your-client-secret>'
od_certificate_password = None  # set only when acl_enabled=True

# OAuth 2.0 credentials (only used if creating a new secret with auth_type=OAUTH2)
od_oauth_client_id     = '<your-client-id>'
od_oauth_client_secret = '<your-client-secret>'
od_oauth_refresh_token = '<your-refresh-token>'

# What to crawl
crawl_personal_drives = True
crawl_shared_with_me  = False   # OAUTH2 only

# Optional filters (leave empty/None to skip)
inclusion_user_email_addresses = []   # e.g. ['alice@contoso.com']
exclusion_user_email_addresses = []
inclusion_drive_items = []            # e.g. ['/drive/root:/Projects']
exclusion_drive_items = []
include_mime_types = []               # e.g. ['application/pdf']
exclude_mime_types = []
absolute_date_after  = None           # ISO 8601, e.g. '2024-01-01T00:00:00Z'
absolute_date_before = None

# Embedding model
# NOTE: The managed default (None) is currently rejected by CreateKnowledgeBase
# in this repo's utility (embeddingModelArn required). Ship with an explicit
# custom model until the utility is fixed. Small extra cost applies.
embedding_model = 'amazon.titan-embed-text-v2:0'
# embedding_model = None  # Managed default (once the utility supports it)

# Cross-region inference profile prefix depends on your region
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:      {region}')
print(f'Account:     {account_id}')
print(f'KB Name:     {knowledge_base_name}')
print(f'Tenant ID:   {tenant_id}')
print(f'Auth:        {auth_type}')
print(f'ACL enabled: {acl_enabled}')
print(f'Embedding:   {embedding_model}')
print(f'Generation:  {generation_model_arn}')

## Step 2 — Create the Secrets Manager secret

Store your OneDrive credentials in AWS Secrets Manager. The key names are dictated by Bedrock — see the AWS docs for [Entra App ID](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-onedrive-entra-setup.html#kb-managed-onedrive-entra-step7) and [OAuth 2.0](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-onedrive-oauth2-setup.html#kb-managed-onedrive-oauth2-step5).

In [ ]:
def _create_or_get_secret(name, secret_dict, region_name):
    sm = boto3.client('secretsmanager', region_name=region_name)
    try:
        resp = sm.create_secret(Name=name, SecretString=json.dumps(secret_dict))
        print(f'  Created secret: {name}')
        return resp['ARN']
    except sm.exceptions.ResourceExistsException:
        resp = sm.describe_secret(SecretId=name)
        print(f'  Secret already exists: {name}')
        return resp['ARN']

if existing_secret_arn:
    secret_arn = existing_secret_arn
    print(f'Using existing secret: {secret_arn}')
elif auth_type == 'ENTRA_APP_ID':
    secret_value = {
        'clientId': od_client_id,
        'clientSecret': od_client_secret,
    }
    if od_certificate_password:
        secret_value['certificatePassword'] = od_certificate_password
    secret_arn = _create_or_get_secret(secret_name, secret_value, region)
elif auth_type == 'OAUTH2':
    secret_value = {
        'clientId': od_oauth_client_id,
        'clientSecret': od_oauth_client_secret,
        'refreshToken': od_oauth_refresh_token,
    }
    secret_arn = _create_or_get_secret(secret_name, secret_value, region)
else:
    raise ValueError(f'Unsupported auth_type: {auth_type}')

print(f'Secret ARN: {secret_arn}')

## Step 3 — Create the Bedrock Managed Knowledge Base

`ManagedKnowledgeBase` handles the KB, IAM role/policies, and the OneDrive data source in one call. The connector params it produces match the [OneDrive schema](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-ds-onedrive-connect.html) documented by AWS.

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

# Build the OneDrive data source configuration
onedrive_config = {
    'type': 'ONEDRIVE',
    'secret_arn': secret_arn,
    'tenant_id': tenant_id,
    'auth_type': auth_type,
    'acl_enabled': acl_enabled,
    'crawl_personal_drives': crawl_personal_drives,
}

if acl_enabled:
    onedrive_config['certificate_s3_path'] = {
        's3BucketName': certificate_s3_bucket,
        's3KeyName': certificate_s3_key,
    }

if auth_type == 'OAUTH2':
    onedrive_config['crawl_shared_with_me'] = crawl_shared_with_me

# Optional filters — only add if set
for k, v in [
    ('inclusion_user_email_addresses', inclusion_user_email_addresses),
    ('exclusion_user_email_addresses', exclusion_user_email_addresses),
    ('inclusion_drive_items', inclusion_drive_items),
    ('exclusion_drive_items', exclusion_drive_items),
    ('include_mime_types', include_mime_types),
    ('exclude_mime_types', exclude_mime_types),
]:
    if v:
        onedrive_config[k] = v
if absolute_date_after:
    onedrive_config['absolute_date_after'] = absolute_date_after
if absolute_date_before:
    onedrive_config['absolute_date_before'] = absolute_date_before

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    data_sources=[onedrive_config],
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_id = kb.kb_id
%store kb_id

## Step 4 — Ingest documents

Bedrock reaches out to Microsoft Graph, enumerates users, and pulls their drive contents.

In [ ]:
time.sleep(30)   # give the DS a moment to reach AVAILABLE
job = kb.start_ingestion_job()

### Check failed documents (optional)

If the ingestion job reports failed files, application logs will explain why. Common causes: OneNote notebooks (unsupported), unsupported binary types, oversized files, per-user permission issues.

In [ ]:
logs_client = boto3.client('logs', region_name=region)
kb_log_group = f'/aws/vendedlogs/bedrock/knowledge-base/APPLICATION_LOGS/{kb.kb_id}'

try:
    end_ms = int(time.time() * 1000)
    start_ms = end_ms - (6 * 60 * 60 * 1000)

    resp = logs_client.filter_log_events(
        logGroupName=kb_log_group,
        startTime=start_ms,
        endTime=end_ms,
        filterPattern='"FAILED"',
        limit=20,
    )
    events = resp.get('events', [])
    if events:
        print(f'=== Failed Documents ({len(events)} events) ===')
        for e in events:
            log = json.loads(e['message'])
            event_data = log.get('event', {})
            doc = event_data.get('document_title', event_data.get('document_id', 'unknown'))
            msg = event_data.get('message', event_data.get('status_reasons', ''))
            print(f'  {doc}: {msg}')
    else:
        print('No failed documents found in logs.')
except logs_client.exceptions.ResourceNotFoundException:
    print(f'Log group not found: {kb_log_group}')
    print('Ingestion logs require enable_logging=True (already set in this notebook).')
except Exception as e:
    print(f'Error checking logs: {e}')

## Step 5 — Query the Knowledge Base

### 5a. Retrieve API

In [ ]:
response = kb.retrieve('What are the main topics across our OneDrive documents?', num_results=5)

print('=== Retrieve API ===')
for i, res in enumerate(response.get('retrievalResults', []), 1):
    score = res['score']
    text = res['content']['text'][:120]
    print(f'{i}. score={score:.4f} | {text}...')
print(f'Total: {len(response.get("retrievalResults", []))} chunks')

### 5b. AgenticRetrieveStream (with generation)

In [ ]:
result = kb.agentic_retrieve_stream(
    query='Summarize the key information from our OneDrive content.',
    model_arn=generation_model_arn,
    generate_response=True,
)

print('=== AgenticRetrieveStream (with generation) ===')
print(f"Answer:\n{result['generated_response']['answer']}")
print(f"\nCitations: {len(result['generated_response'].get('citations', []))}")

### 5c. AgenticRetrieveStream API

In [ ]:
result = kb.agentic_retrieve_stream(
    query='What projects are documented across users\' drives and how do they relate?',
    model_arn=generation_model_arn,
    max_results=10,
    max_iterations=3,
)

print('=== AgenticRetrieveStream API ===')
print(f'Trace events: {len(result["traces"])}')
print(f'Final: {len(result["results"])} deduplicated chunks')
for i, r in enumerate(result['results'][:5], 1):
    print(f'  {i}. {r["content"]["text"][:120]}...')
if result.get('generated_response'):
    print(f"\nGenerated Answer:\n{result['generated_response']['answer']}")
    print(f"Citations: {len(result['generated_response'].get('citations', []))}")

### 5d. Try your own queries

In [ ]:
# Try your own query
QUERY = 'What is the latest update across our OneDrive content?'

result = kb.agentic_retrieve_stream(
    query=QUERY,
    model_arn=generation_model_arn,
    generate_response=True,
)
print(result['generated_response']['answer'])

## Step 6 — Add more data sources (optional)

In [ ]:
# Uncomment to add an S3 data source to this KB
# s3_ds_id = kb.add_data_source({
#     'type': 'S3',
#     'bucket_name': f'bedrock-bmkb-docs-{account_id}',
# })
# kb.start_ingestion_job(ds_id=s3_ds_id)
# print(f'Data sources: {kb.data_sources}')

## Step 7 — Cleanup

> Only run this when you're done experimenting.

In [ ]:
# Uncomment to delete everything
print('===============================Deleting Knowledge Base and associated resources==============================')
# kb.delete_kb(delete_iam=True)
#
# try:
#     boto3.client('secretsmanager', region_name=region).delete_secret(
#         SecretId=secret_name, ForceDeleteWithoutRecovery=True
#     )
#     print(f'Deleted secret: {secret_name}')
# except Exception as e:
#     print(f'Error deleting secret: {e}')

## Summary

| What | Details |
|---|---|
| KB Type | `MANAGED` — Bedrock handles the vector store |
| Data Source | OneDrive via `MANAGED_KNOWLEDGE_BASE_CONNECTOR` |
| Auth | `ENTRA_APP_ID` (recommended) or `OAUTH2` |
| Parsing | Smart Parsing (default) |
| Embedding | Managed default (no extra cost) or custom |
| IAM | Auto-created role + policies (model, CloudWatch, Secrets Manager) |

### OneDrive configuration options

| Parameter | Description |
|-----------|-------------|
| `tenant_id` | Microsoft Entra (Azure AD) tenant ID |
| `auth_type` | `ENTRA_APP_ID` or `OAUTH2` |
| `acl_enabled` | Document-level access control (requires `ENTRA_APP_ID` + certificate). Immutable after creation. |
| `certificate_s3_path` | `{s3BucketName, s3KeyName}` for the `.p12` bundle (ACLs only) |
| `crawl_personal_drives` | Whether to crawl users' personal drives (default `True`) |
| `crawl_shared_with_me` | Whether to crawl files shared with the signed-in user (`OAUTH2` only) |
| `inclusion_user_email_addresses` / `exclusion_user_email_addresses` | Per-user filters |
| `user_filter_path` | S3 URL of a file listing users to include/exclude |
| `inclusion_drive_items` / `exclusion_drive_items` | Drive-item path filters |
| `include_mime_types` / `exclude_mime_types` | MIME-type filters |
| `absolute_date_after` / `absolute_date_before` | ISO 8601 date filters |

> **Note:** OneNote notebooks are not currently supported by the OneDrive connector.